# Breaking Defenses & Black-Box Attacks

In [1]:
import torch
from torch import nn
from torch.optim import Adam
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader

from torchvision import transforms
from torchvision.models import resnet18, mobilenet_v2
from torchvision.models import ResNet18_Weights
from torchvision.datasets.cifar import CIFAR10

from tqdm import trange, tqdm

torch.manual_seed(0)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

# CIFAR10 Dataset (5 points)

In [2]:
norm_mean = (0.4914, 0.4822, 0.4465)
norm_std = (0.2023, 0.1994, 0.2010)
batch_size = 128

mu = torch.tensor(norm_mean).view(3,1,1).to(device)
std = torch.tensor(norm_std).view(3,1,1).to(device)
scale = 1 / std.mean().item()

# TODO: Set the upper limit and lower limit possible for images
upper_limit = ((1 - mu) / std)
lower_limit = ((0 - mu) / std)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(norm_mean, norm_std),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(norm_mean, norm_std),
])

trainset = CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)

testset = CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)


classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')


100%|██████████| 170M/170M [00:05<00:00, 28.6MB/s] 


# Defensive Distillation (25 points)

[Defensive distillation](https://arxiv.org/abs/1511.04508) proceeds in four steps:

1.   **Train the teacher network**, by setting the temperature of the softmax to T during the
training phase.
2.   **Compute soft labels** by apply the teacher network to each instance in the training set, again evaluating the softmax at temperature T.
3.  **Train the distilled network** (a network with the same shape as the teacher network) on the soft labels, using softmax at temperature T.
4.  Finally, when running the distilled network at test time to classify new inputs, use temperature 1.



## Train the teacher

In [3]:
def train_step(model, dataloader, loss_fn, optimizer, temperature):
    # TODO: Return loss and accuracy for each epoch
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        logits = model(x)
        
        # Apply temperature to logits
        loss = loss_fn(logits / temperature, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = logits.max(1)
        total += y.size(0)
        correct += predicted.eq(y).sum().item()
    
    avg_loss = total_loss / len(dataloader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy


def train_teacher(model, n_epochs, loader=trainloader, temp=100):
    # TODO: Log the accuracy and loss for each epoch
    model.to(device)
    optimizer = Adam(model.parameters(), lr=0.001)
    loss_fn = CrossEntropyLoss()
    
    for epoch in range(n_epochs):
        loss, acc = train_step(model, loader, loss_fn, optimizer, temp)
        print(f'Epoch [{epoch+1}/{n_epochs}] - Loss: {loss:.4f}, Accuracy: {acc:.2f}%')

You can use a pre-trained resnet to speed up the training process.

In [ ]:
teacher = resnet18(weights=ResNet18_Weights.DEFAULT)
teacher.fc = nn.Linear(teacher.fc.in_features, 10)
teacher.layer4 = nn.Sequential(
    nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(512),
    nn.ReLU(inplace=True),
    nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=False),
    nn.BatchNorm2d(512),
    nn.ReLU(inplace=True)
)

train_teacher(teacher, 15)

Epoch [1/15] - Loss: 1.7023, Accuracy: 64.31%
Epoch [2/15] - Loss: 0.9058, Accuracy: 74.06%
Epoch [3/15] - Loss: 0.6813, Accuracy: 78.18%
Epoch [4/15] - Loss: 0.5993, Accuracy: 80.37%
Epoch [5/15] - Loss: 0.5385, Accuracy: 82.00%
Epoch [6/15] - Loss: 0.5036, Accuracy: 83.09%
Epoch [7/15] - Loss: 0.4737, Accuracy: 83.96%
Epoch [8/15] - Loss: 0.4503, Accuracy: 84.72%
Epoch [9/15] - Loss: 0.4243, Accuracy: 85.58%
Epoch [10/15] - Loss: 0.4082, Accuracy: 86.22%
Epoch [11/15] - Loss: 0.3904, Accuracy: 86.79%
Epoch [12/15] - Loss: 0.3756, Accuracy: 87.12%
Epoch [13/15] - Loss: 0.3627, Accuracy: 87.76%
Epoch [14/15] - Loss: 0.3490, Accuracy: 87.96%
Epoch [15/15] - Loss: 0.3346, Accuracy: 88.47%


## Test the teacher

In [4]:
def test_clean(model, dataloader=testloader):
    # TODO: Return the clean accuracy of the model
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            _, predicted = logits.max(1)
            total += y.size(0)
            correct += predicted.eq(y).sum().item()
    
    accuracy = 100. * correct / total
    return accuracy

Print the clean accuracy of the teacher.

In [19]:
print(f'Teacher Accuracy {test_clean(teacher):.2f}%')

Teacher Accuracy 83.99%


## Train the student

In [5]:
def distill(model, teacher, dataloader, optimizer, T):
    # TODO: Get soft labels from teacher model
    # TODO: Get student model outputs
    # TODO: Compute the distillation loss
    # TODO: Return the accuracy (on real labels) and loss (on soft labels)
    model.train()
    teacher.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        
        # Get soft labels from teacher
        with torch.no_grad():
            teacher_logits = teacher(x)
            soft_labels = F.softmax(teacher_logits / T, dim=1)
        
        # Get student outputs
        student_logits = model(x)
        student_soft = F.log_softmax(student_logits / T, dim=1)
        
        # Compute distillation loss (KL divergence)
        loss = F.kl_div(student_soft, soft_labels, reduction='batchmean')
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = student_logits.max(1)
        total += y.size(0)
        correct += predicted.eq(y).sum().item()
    
    avg_loss = total_loss / len(dataloader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy


def train_student(model, teacher, n_epochs, loader=trainloader, temp=100):
    # TODO: Log the accuracy and loss for each epoch
    model.to(device)
    teacher.to(device)
    optimizer = Adam(model.parameters(), lr=0.01)
    
    for epoch in range(n_epochs):
        loss, acc = distill(model, teacher, loader, optimizer, temp)
        print(f'Epoch [{epoch+1}/{n_epochs}] - Loss: {loss:.4f}, Accuracy: {acc:.2f}%')

This time use a `resnet18` without the pretrained weights.

In [26]:
student = resnet18(weights=None)
student.fc = nn.Linear(student.fc.in_features, 10)

train_student(student, teacher, 15)

Epoch [1/15] - Loss: 1.3081, Accuracy: 36.99%
Epoch [2/15] - Loss: 0.9275, Accuracy: 54.01%
Epoch [3/15] - Loss: 0.7167, Accuracy: 62.72%
Epoch [4/15] - Loss: 0.5944, Accuracy: 67.65%
Epoch [5/15] - Loss: 0.5123, Accuracy: 70.63%
Epoch [6/15] - Loss: 0.4467, Accuracy: 73.47%
Epoch [7/15] - Loss: 0.4051, Accuracy: 75.31%
Epoch [8/15] - Loss: 0.3709, Accuracy: 76.48%
Epoch [9/15] - Loss: 0.3435, Accuracy: 77.69%
Epoch [10/15] - Loss: 0.3240, Accuracy: 78.45%
Epoch [11/15] - Loss: 0.3079, Accuracy: 79.06%
Epoch [12/15] - Loss: 0.2862, Accuracy: 79.72%
Epoch [13/15] - Loss: 0.2764, Accuracy: 80.56%
Epoch [14/15] - Loss: 0.2634, Accuracy: 81.04%
Epoch [15/15] - Loss: 0.2538, Accuracy: 81.56%


## Test the student

In [27]:
print(f'Student Accuracy {test_clean(student):.2f}%')

Student Accuracy 79.69%


# Attack (15 points)

Implement the FGSM attack and the `test_attack` funcion to report the robust accuracy for different values of epsilon.

In [6]:
def attack_fgsm(model, x, y, epsilon):
    # TODO: Return perturbed input
    x_adv = x.clone().detach().requires_grad_(True)
    
    logits = model(x_adv)
    loss = F.cross_entropy(logits, y)
    loss.backward()
    
    # Get gradient sign
    grad_sign = x_adv.grad.sign()
    
    # Create adversarial example
    x_adv = x_adv + epsilon * grad_sign
    x_adv = torch.clamp(x_adv, lower_limit, upper_limit)
    
    return x_adv.detach()


def attack_pgd(model, x, y, epsilon, alpha=0.2, num_iters=10):
    # TODO: Return perturbed input
    x_adv = x.clone().detach()
    
    for i in range(num_iters):
        x_adv.requires_grad_(True)
        
        logits = model(x_adv)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        
        # Update adversarial example
        grad_sign = x_adv.grad.sign()
        x_adv = x_adv.detach() + alpha * epsilon * grad_sign
        
        # Project back to epsilon ball
        x_adv = torch.max(torch.min(x_adv, x + epsilon), x - epsilon)
        x_adv = torch.clamp(x_adv, lower_limit, upper_limit)
    
    return x_adv.detach()


def test_attack(model, epsilon, atttack=attack_fgsm, loader=testloader):
    # TODO: Return the robust accuracy for FGSM or PGD
    model.eval()
    correct = 0
    total = 0
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        
        # Generate adversarial examples
        x_adv = atttack(model, x, y, epsilon)
        
        # Test on adversarial examples
        with torch.no_grad():
            logits = model(x_adv)
            _, predicted = logits.max(1)
            total += y.size(0)
            correct += predicted.eq(y).sum().item()
    
    accuracy = 100. * correct / total
    return accuracy

Report the robust accuracy of the teacher for `ϵ = [1, 2, 4, 8, 16]`.

In [30]:
epsilons = [1, 2, 4, 8, 16]

for eps in epsilons:
    # TODO:
    acc = test_attack(teacher, eps/255, atttack=attack_fgsm)
    print(f'FGSM with ϵ={eps}/255 has Accuracy: {acc:.2f}%')
    acc = test_attack(teacher, eps/255, atttack=attack_pgd)
    print(f'PGD  with ϵ={eps}/255 has Accuracy: {acc:.2f}%')

FGSM with ϵ=1/255 has Accuracy: 80.20%
PGD  with ϵ=1/255 has Accuracy: 79.83%
FGSM with ϵ=2/255 has Accuracy: 79.36%
PGD  with ϵ=2/255 has Accuracy: 79.19%
FGSM with ϵ=4/255 has Accuracy: 79.16%
PGD  with ϵ=4/255 has Accuracy: 79.05%
FGSM with ϵ=8/255 has Accuracy: 79.08%
PGD  with ϵ=8/255 has Accuracy: 79.01%
FGSM with ϵ=16/255 has Accuracy: 79.08%
PGD  with ϵ=16/255 has Accuracy: 79.01%


Do the same for the student:

In [31]:
for eps in epsilons:
    # TODO:
    acc = test_attack(student, eps/255, atttack=attack_fgsm)
    print(f'FGSM with ϵ={eps}/255 has Accuracy: {acc:.2f}%')
    acc = test_attack(student, eps/255, atttack=attack_pgd)
    print(f'PGD  with ϵ={eps}/255 has Accuracy: {acc:.2f}%')

FGSM with ϵ=1/255 has Accuracy: 76.27%
PGD  with ϵ=1/255 has Accuracy: 75.84%
FGSM with ϵ=2/255 has Accuracy: 73.76%
PGD  with ϵ=2/255 has Accuracy: 73.13%
FGSM with ϵ=4/255 has Accuracy: 72.04%
PGD  with ϵ=4/255 has Accuracy: 71.69%
FGSM with ϵ=8/255 has Accuracy: 71.53%
PGD  with ϵ=8/255 has Accuracy: 71.33%
FGSM with ϵ=16/255 has Accuracy: 71.42%
PGD  with ϵ=16/255 has Accuracy: 71.25%


What do you see?

`your response: The attacks don't work for them, as they are both trained with T=100. Of course if we set the temprature back at 100, the attacks would work.` 

# Transferring Adversarial Examples (15 points)

Train yet another model to be used as the surrogate. (set temperature to 1)

In [13]:
surrogate = resnet18(weights=ResNet18_Weights.DEFAULT)
surrogate.fc = nn.Linear(surrogate.fc.in_features, 10)
surrogate.layer4 = nn.Sequential(
    nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(512),
    nn.ReLU(inplace=True),
    nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=False),
    nn.BatchNorm2d(512),
    nn.ReLU(inplace=True)
)

train_teacher(surrogate, 15, temp=1)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 164MB/s] 


Epoch [1/15] - Loss: 0.9028, Accuracy: 68.74%
Epoch [2/15] - Loss: 0.6832, Accuracy: 76.53%
Epoch [3/15] - Loss: 0.5972, Accuracy: 79.33%
Epoch [4/15] - Loss: 0.5529, Accuracy: 80.87%
Epoch [5/15] - Loss: 0.5168, Accuracy: 82.13%
Epoch [6/15] - Loss: 0.4841, Accuracy: 83.45%
Epoch [7/15] - Loss: 0.4598, Accuracy: 84.21%
Epoch [8/15] - Loss: 0.4400, Accuracy: 84.83%
Epoch [9/15] - Loss: 0.4189, Accuracy: 85.40%
Epoch [10/15] - Loss: 0.4096, Accuracy: 85.81%
Epoch [11/15] - Loss: 0.3904, Accuracy: 86.54%
Epoch [12/15] - Loss: 0.3702, Accuracy: 87.09%
Epoch [13/15] - Loss: 0.3615, Accuracy: 87.48%
Epoch [14/15] - Loss: 0.3465, Accuracy: 88.05%
Epoch [15/15] - Loss: 0.3376, Accuracy: 88.38%


Print the surrogate accuracy.

In [44]:
print(f'Surrogate Accuracy {test_clean(surrogate):.2f}%')

Surrogate Accuracy 84.54%


Report the accuracy of the surrogate for `ϵ = [1, 2, 4, 8, 16]`.

In [45]:
for eps in epsilons:
    # TODO:
    acc = test_attack(surrogate, eps/255, atttack=attack_fgsm)
    print(f'FGSM with ϵ={eps}/255 has Accuracy: {acc:.2f}%')
    acc = test_attack(surrogate, eps/255, atttack=attack_pgd)
    print(f'PGD  with ϵ={eps}/255 has Accuracy: {acc:.2f}%')

FGSM with ϵ=1/255 has Accuracy: 79.48%
PGD  with ϵ=1/255 has Accuracy: 79.27%
FGSM with ϵ=2/255 has Accuracy: 73.70%
PGD  with ϵ=2/255 has Accuracy: 72.59%
FGSM with ϵ=4/255 has Accuracy: 62.31%
PGD  with ϵ=4/255 has Accuracy: 57.49%
FGSM with ϵ=8/255 has Accuracy: 42.52%
PGD  with ϵ=8/255 has Accuracy: 29.90%
FGSM with ϵ=16/255 has Accuracy: 20.43%
PGD  with ϵ=16/255 has Accuracy: 4.76%


Implement the following functions to transfer attacks from a surrogate model to an oracle.

In [7]:
def transfer_attack(oracle, model, eps, loader=testloader):
    # TODO: Attack the model and report the accuracy of the oracle
    model.eval()
    oracle.eval()
    correct = 0
    total = 0
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        
        # Generate adversarial examples on the surrogate model
        x_adv = attack_fgsm(model, x, y, eps)
        
        # Test on the oracle model
        with torch.no_grad():
            logits = oracle(x_adv)
            _, predicted = logits.max(1)
            total += y.size(0)
            correct += predicted.eq(y).sum().item()
    
    accuracy = 100. * correct / total
    return accuracy

Transfer attacks for `ϵ = [1, 2, 4, 8, 16]` from your model to the student.

In [ ]:
for eps in epsilons:
    acc = transfer_attack(student, surrogate, eps*scale/255)
    print(f'FGSM with ϵ={eps}/255 has Accuracy: {acc:.2f}%')

FGSM with ϵ=1/255 has Accuracy: 76.23%
FGSM with ϵ=2/255 has Accuracy: 72.58%
FGSM with ϵ=4/255 has Accuracy: 64.49%
FGSM with ϵ=8/255 has Accuracy: 48.69%
FGSM with ϵ=16/255 has Accuracy: 27.66%


- What can be inferred from these results?
- How are the accuracies of the student and the surrogate under attack related?
- Does Defensive Distillation obfuscate the gradients? Why?

`your response: Though the gradiants don't flow to make adv examples on the actual models, if you train a healthy (t=1) model and find edverserial examples on it, they will almost definitely transfer to the teacher and student. The student and surrogate are closely related and attacks on surrogate work on student too. Because the student is not really more robust, it just has a mechanism which makes its outputs nearly onehot and therefore in usual attacks, gradients don't go past that layer. However, if only we used temperature=100 in the test time, we could attack the student directly, and if not possible, as you can see, we can still attack the student using transfer learning. And yes because of all of what I just said, DD is using gradiant obfuscation.`


# ZOO Based Black-Box Attacks (25 points)

Based on [Black-box Adversarial Attacks with Limited Queries and Information](https://arxiv.org/abs/1804.08598) you must first calculate the estimate of the graidents, and next attack the model based on your estimates.

In [8]:
def nes_gradient_estimate(model, x, y, epsilon, num_samples, sigma):
    model.eval()
    grad_estimate = torch.zeros_like(x)
    
    for i in range(num_samples):
        # Sample random noise
        delta = torch.randn_like(x)
        
        # Query model at perturbed points
        with torch.no_grad():
            # Positive perturbation
            x_pos = x + sigma * delta
            logits_pos = model(x_pos)
            probs_pos = F.softmax(logits_pos, dim=1)
            loss_pos = -probs_pos[torch.arange(len(y)), y]
            
            # Negative perturbation
            x_neg = x - sigma * delta
            logits_neg = model(x_neg)
            probs_neg = F.softmax(logits_neg, dim=1)
            loss_neg = -probs_neg[torch.arange(len(y)), y]
            
            # NES gradient estimate with proper broadcasting
            # loss_pos/neg: [batch_size], delta: [batch_size, C, H, W]
            grad_estimate += (loss_pos - loss_neg).view(-1, 1, 1, 1) * delta
    
    grad_estimate = grad_estimate / (2 * num_samples * sigma)
    return grad_estimate

I used 3 different things to estimate gradiant and all of them end up almost the same result. The bottom result is made with probabilities.

In [9]:
def partial_information_attack(model, x, y, epsilon, num_samples, sigma, num_steps, alpha):
    # TODO: Return the perturbed image
    x_adv = x.clone().detach()
    
    for step in range(num_steps):
        # Estimate gradient using NES
        grad = nes_gradient_estimate(model, x_adv, y, epsilon, num_samples, sigma)
        
        # PGD update with gradient sign
        x_adv = x_adv + alpha * epsilon * grad.sign()
        
        # Project back to epsilon ball
        x_adv = torch.max(torch.min(x_adv, x + epsilon), x - epsilon)
        x_adv = torch.clamp(x_adv, lower_limit, upper_limit)
    
    return x_adv.detach()

Now run this attack on your models and report the results. (You **DON'T** need to run the attack for the entire test dataset as this will take a lot of time!)

In [10]:
def test_zoo_attack(model, epsilon, num_samples, sigma, num_steps, alpha, loader=testloader, num_batches=None):
    model.eval()
    correct = 0
    total = 0

    for batch_idx, (x, y) in enumerate(tqdm(loader)):
        if num_batches and num_batches <= batch_idx:
            break
        x, y = x.to(device), y.to(device)
        
        # Generate adversarial examples using ZOO
        x_adv = partial_information_attack(model, x, y, epsilon, num_samples, sigma, num_steps, alpha)
        
        # Test on adversarial examples
        with torch.no_grad():
            logits = model(x_adv)
            _, predicted = logits.max(1)
            total += y.size(0)
            correct += predicted.eq(y).sum().item()
    
    accuracy = 100. * correct / total
    return accuracy

In [19]:
epsilons = [1, 2, 4, 8, 16]

num_batches = 10 # it takes too long. With 10 the pattern is clear too.
for eps in epsilons:
    acc = test_zoo_attack(model=surrogate, epsilon=eps*scale/255, num_samples=100, sigma=0.001, num_steps=10, alpha=0.1, loader=testloader, num_batches=num_batches)
    print(f'ZOO with ϵ={eps}/255 has Accuracy: {acc:.2f}%')

 13%|█▎        | 10/79 [01:42<11:47, 10.25s/it]


ZOO with ϵ=1/255 has Accuracy: 80.31%


 13%|█▎        | 10/79 [01:42<11:47, 10.26s/it]


ZOO with ϵ=2/255 has Accuracy: 73.67%


 13%|█▎        | 10/79 [01:42<11:47, 10.25s/it]


ZOO with ϵ=4/255 has Accuracy: 60.23%


 13%|█▎        | 10/79 [01:42<11:47, 10.26s/it]


ZOO with ϵ=8/255 has Accuracy: 35.31%


 13%|█▎        | 10/79 [01:42<11:47, 10.26s/it]

ZOO with ϵ=16/255 has Accuracy: 10.62%


# Adversarially Robust Distillation (15 points)

In this section we are going to test another type of distillation to see if this method is robust. This technique is [Adversarially Robust Distillation](https://arxiv.org/abs/1905.09747).



1.   We will try to distill a robsut teacher from [Robust Bench](https://robustbench.github.io/) onto a smaller architecture.
2.   We minimize the KL-Divergence between the logits of the student and teacher to ensure fidelity. (You can also incorporate the classification loss as mentioned in the paper but you can choose to ignore it as well)
3.   At each step of the distillation you will attack the student (you can use either FGSM or PGD) and find an adversarial example $X + \delta$ for data point $X$. Next you will minimize $t^2 \times \text{KL}(S(X+\delta), T(X))$ where $S$ and $T$ are the student and teacher networks respectively.



In [11]:
! pip install git+https://github.com/RobustBench/robustbench.git

  Cloning https://github.com/RobustBench/robustbench.git to /tmp/pip-req-build-ufuh22gi
  Running command git clone --filter=blob:none --quiet https://github.com/RobustBench/robustbench.git /tmp/pip-req-build-ufuh22gi
  Resolved https://github.com/RobustBench/robustbench.git to commit 78fcc9e48a07a861268f295a777b975f25155964
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/fra31/auto-attack.git (to revision a39220048b3c9f2cca9a4d3a54604793c68eca7e) to /tmp/pip-install-er8kcrou/autoattack_2dafcdc70eef4fd990316134d981ca95
  Running command git clone --filter=blob:none --quiet https://github.com/fra31/auto-attack.git /tmp/pip-install-er8kcrou/autoattack_2dafcdc70eef4fd990316134d981ca95
  Running command git rev-parse -q --verify 'sha^a39220048b3c9f2cca9a4d3a54604793c68eca7e'
  Running command git fetch -q https://github.com/fra31/auto-attack.git a39220048b3c9f2cca9a4d3a54604793c68eca7e
  Resolved https://github.com/fra31/auto-attack.git to commit a39220048b3c9f2cca9a4

In [12]:
from robustbench.utils import load_model

teacher = load_model(model_name='Gowal2021Improving_R18_ddpm_100m', dataset='cifar10', threat_model='Linf')

Downloading...
From (original): https://drive.google.com/uc?id=1-0EuCJashqSXEkkd1DOzFA4tH8KL2kim
From (redirected): https://drive.google.com/uc?id=1-0EuCJashqSXEkkd1DOzFA4tH8KL2kim&confirm=t&uuid=790e2abf-9daa-463f-895c-60a65f1409e6
To: /content/models/cifar10/Linf/Gowal2021Improving_R18_ddpm_100m.pt
100%|██████████| 50.3M/50.3M [00:00<00:00, 64.2MB/s]


In [19]:
from torch.optim import SGD

def ard(student, teacher, dataloader, optimizer, eps, attack, T=1, beta=0.3):
    student.train()
    teacher.eval()
    total_loss = 0
    total_kl_loss = 0
    total_ce_loss = 0
    correct = 0
    total = 0
    
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        
        # Generate adversarial examples for student
        # Note: Paper uses PGD attack on the student model
        x_adv = attack(student, x, y, eps)
        
        # KEY: Teacher predicts on CLEAN images
        with torch.no_grad():
            teacher_logits = teacher(x)  # Clean images!
            teacher_soft = F.softmax(teacher_logits / T, dim=1)
        
        # Student predicts on ADVERSARIAL images
        student_logits = student(x_adv)  # Adversarial images!
        student_soft = F.log_softmax(student_logits / T, dim=1)
        
        # KL divergence loss (distillation loss)
        kl_loss = F.kl_div(student_soft, teacher_soft, reduction='batchmean')
        
        # Hard label loss (cross-entropy with true labels)
        ce_loss = F.cross_entropy(student_logits, y)
        
        # Combined loss: ARD = KL divergence + beta * CE
        # The T^2 scaling is absorbed into the beta hyperparameter
        loss = kl_loss + beta * ce_loss
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_kl_loss += kl_loss.item()
        total_ce_loss += ce_loss.item()
        _, predicted = student_logits.max(1)
        total += y.size(0)
        correct += predicted.eq(y).sum().item()
    
    avg_loss = total_loss / len(dataloader)
    avg_kl = total_kl_loss / len(dataloader)
    avg_ce = total_ce_loss / len(dataloader)
    accuracy = 100. * correct / total
    
    return avg_loss, avg_kl, avg_ce, accuracy


def adv_train_student(model, teacher, n_epochs, eps=8/255, loader=trainloader):
    model.to(device)
    teacher.to(device)
    
    # Use SGD with momentum (more stable than Adam for this task)
    # Paper uses learning rate 0.1 with decay
    optimizer = Adam(model.parameters(), lr=0.004)
    
    for epoch in range(n_epochs):
        loss, kl_loss, ce_loss, acc = ard(
            model, teacher, loader, optimizer, eps, attack_pgd, T=1, beta=0.3
        )
        
        print(f'Epoch [{epoch+1}/{n_epochs}]')
        print(f'  Total Loss: {loss:.4f} (KL: {kl_loss:.4f}, CE: {ce_loss:.4f})')
        print(f'  Accuracy: {acc:.2f}%')

In [20]:
student = mobilenet_v2(weights=None)

# TODO: Adjust and train the student
student.classifier[1] = nn.Linear(student.classifier[1].in_features, 10)

adv_train_student(student, teacher, 10) # Ran out of GPU:)

Epoch [1/10]
  Total Loss: 2.2341 (KL: 1.5973, CE: 2.1226)
  Accuracy: 19.93%
Epoch [2/10]
  Total Loss: 2.1960 (KL: 1.6457, CE: 1.8344)
  Accuracy: 30.23%
Epoch [3/10]
  Total Loss: 2.2028 (KL: 1.6870, CE: 1.7194)
  Accuracy: 35.57%
Epoch [4/10]
  Total Loss: 2.2184 (KL: 1.7276, CE: 1.6359)
  Accuracy: 39.94%
Epoch [5/10]
  Total Loss: 2.2336 (KL: 1.7677, CE: 1.5530)
  Accuracy: 43.53%
Epoch [6/10]
  Total Loss: 2.2449 (KL: 1.8013, CE: 1.4786)
  Accuracy: 46.03%
Epoch [7/10]
  Total Loss: 2.2458 (KL: 1.8191, CE: 1.4225)
  Accuracy: 48.38%
Epoch [8/10]
  Total Loss: 2.2535 (KL: 1.8376, CE: 1.3862)
  Accuracy: 49.85%
Epoch [9/10]
  Total Loss: 2.2592 (KL: 1.8553, CE: 1.3465)
  Accuracy: 51.30%
Epoch [10/10]
  Total Loss: 2.2603 (KL: 1.8629, CE: 1.3247)
  Accuracy: 52.08%


Now report the accuracy of the student on the test dataset.

In [22]:
# TODO: Clean accurcy
print(f'Student Accuracy {test_clean(student):.2f}%')
# TODO: FGSM with eps=8/255
eps=8
acc = test_attack(student, eps * scale / 255, atttack=attack_fgsm)
print(f'FGSM with ϵ={eps}/255 has Accuracy: {acc:.2f}%')
# TODO: PGD with eps=8/255
acc = test_attack(student, eps * scale / 255, atttack=attack_pgd)
print(f'PGD  with ϵ={eps}/255 has Accuracy: {acc:.2f}%')

Student Accuracy 68.12%
FGSM with ϵ=8/255 has Accuracy: 18.18%
PGD  with ϵ=8/255 has Accuracy: 9.81%
